# Banking77 Intent Classification with Unsloth

**Course:** CSC15012 – Applications of Natural Language Processing in Industry  
**Student:** Lê Đình Minh Quân – 23127460  
**University:** HCMUS – VNUHCM

---

### Autopilot Notebook – Run all cells from top to bottom

| Step | Description |
|------|-------------|
| 1 | Install dependencies |
| 2 | Create / sync project structure & files |
| 3 | Preprocess BANKING77 dataset |
| 4 | Fine-tune LLaMA 3.2 3B (QLoRA via Unsloth) |
| 5 | Evaluate model & demo inference |
| 6 | Update README and push clean code to GitHub |

> **Runtime:** Use a **GPU** runtime (H100 / A100 / T4). Go to `Runtime -> Change runtime type -> GPU`.  
> **Secrets:** Store your GitHub PAT as **`GITHUB_PAT`** in Colab Secrets (key icon, left sidebar).  
> **Optional:** If you want the final README to contain your public demo video link, update `VIDEO_LINK` in **Step 1** before the last GitHub push cell.


In [1]:
# ============================================================
# Step 0: Install dependencies, set cache dirs, set HF token & verify GPU
# ============================================================
!nvidia-smi
print()

!pip install -q unsloth
!pip install -q datasets pandas scikit-learn pyyaml

import os
import importlib.metadata as importlib_metadata

# Keep large caches OUTSIDE the git repo
os.environ.setdefault("HF_HOME", "/content/hf_cache")
os.environ.setdefault("HUGGINGFACE_HUB_CACHE", "/content/hf_cache/hub")
os.environ.setdefault("HF_DATASETS_CACHE", "/content/hf_cache/datasets")
os.environ.setdefault("TRANSFORMERS_CACHE", "/content/hf_cache/transformers")
os.environ.setdefault("XDG_CACHE_HOME", "/content/.cache")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

for cache_dir in [
    os.environ["HF_HOME"],
    os.environ["HUGGINGFACE_HUB_CACHE"],
    os.environ["HF_DATASETS_CACHE"],
    os.environ["TRANSFORMERS_CACHE"],
    os.environ["XDG_CACHE_HOME"],
]:
    os.makedirs(cache_dir, exist_ok=True)

# Set HuggingFace token (optional, but helps avoid rate limits)
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN set from Colab Secrets")
except Exception:
    print("No HF_TOKEN found (optional – public datasets work without it)")

import torch
print(f"\nCUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

def _pkg_ver(name: str) -> str:
    try:
        return importlib_metadata.version(name)
    except Exception:
        return "not found"

print("\nKey package versions:")
for pkg in ["unsloth", "transformers", "trl", "datasets", "torch", "xformers"]:
    print(f"  - {pkg}: {_pkg_ver(pkg)}")

print("\nDependencies installed!")


Mon Apr 27 17:00:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          Off |   00000000:04:00.0 Off |                    0 |
| N/A   28C    P0             68W /  700W |       0MiB /  81559MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [13]:
# ============================================================
# Step 1: Create project structure & write config/utility files
# ============================================================
import os
import re
import json
import shutil
import textwrap
import subprocess

PROJECT = "/content/banking-intent-unsloth"
GITHUB_USER = "ledinhminhquan"
REPO_NAME = "banking-intent-unsloth"
PUBLIC_REPO_URL = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"

# Update these 2 links once before your FINAL push
NOTEBOOK_LINK = "https://drive.google.com/file/d/1dd4BxqryUvpkOvmDkm8-ZWor0LHpVSZu/view?usp=sharing"
VIDEO_LINK = "https://drive.google.com/file/d/1D9hAiMirLMdjMijR-znzDAdYw2LGtdqP/view?usp=sharing"


# Important for re-runs in the SAME Colab session: move out of the repo folder first
os.chdir("/content")

# Fresh local folder each run; if repo already exists, clone it first to preserve git history
if os.path.exists(PROJECT):
    shutil.rmtree(PROJECT)

clone_result = subprocess.run(
    ["git", "clone", PUBLIC_REPO_URL, PROJECT],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
)

if clone_result.returncode == 0:
    print("Cloned existing GitHub repo to preserve history.")
else:
    os.makedirs(PROJECT, exist_ok=True)
    print("GitHub repo not found yet (or clone unavailable). Starting from a fresh local project folder.")

for d in ["scripts", "configs", "sample_data"]:
    os.makedirs(f"{PROJECT}/{d}", exist_ok=True)

%cd /content/banking-intent-unsloth

def render_readme(notebook_link: str = None, video_link: str = None):
    """Create README.md using current files and (if present) evaluation_results.json."""
    if notebook_link is None:
        notebook_link = NOTEBOOK_LINK
    if video_link is None:
        video_link = VIDEO_LINK

    import pandas as pd

    metrics = {}
    if os.path.exists("evaluation_results.json"):
        with open("evaluation_results.json", encoding="utf-8") as f:
            metrics = json.load(f)

    train_rows = None
    test_rows = None
    num_labels = None

    if os.path.exists("sample_data/train.csv"):
        train_rows = len(pd.read_csv("sample_data/train.csv"))
    if os.path.exists("sample_data/test.csv"):
        test_rows = len(pd.read_csv("sample_data/test.csv"))
    if os.path.exists("sample_data/label_map.json"):
        with open("sample_data/label_map.json", encoding="utf-8") as f:
            label_map = json.load(f)
        num_labels = label_map.get("num_labels")

    def fmt_pct(x):
        return f"{100 * x:.2f}%" if isinstance(x, (int, float)) else "_TBD_"

    def fmt_num(x, digits=4):
        return f"{x:.{digits}f}" if isinstance(x, (int, float)) else "_TBD_"

    accuracy = fmt_pct(metrics.get("accuracy"))
    macro_precision = fmt_num(metrics.get("macro_precision"), digits=2)
    macro_recall = fmt_num(metrics.get("macro_recall"), digits=2)
    macro_f1 = fmt_num(metrics.get("macro_f1"), digits=2)
    training_loss = fmt_num(metrics.get("training_loss"), digits=4)
    train_seconds = metrics.get("training_time_seconds")
    training_time = f"{train_seconds:.1f}s (~{train_seconds/60:.1f} min)" if isinstance(train_seconds, (int, float)) else "_TBD_"

    train_rows_text = str(train_rows) if train_rows is not None else "_TBD_"
    test_rows_text = str(test_rows) if test_rows is not None else "_TBD_"
    num_labels_text = str(num_labels) if num_labels is not None else "_TBD_"

    video_line = (
        f"📹 **Video link**: [Google Drive – Demo Video]({video_link})"
        if video_link and video_link != "PASTE_PUBLIC_VIDEO_LINK_HERE"
        else "📹 **Video link**: _Update this link after recording and uploading your public demo video._"
    )

    notebook_line = (
        f"📓 **Notebook link**: [Google Drive / Colab Notebook]({notebook_link})"
        if notebook_link
        else "📓 **Notebook link**: _Add your Colab notebook link here._"
    )

    readme = f'''# Banking77 Intent Classification with Unsloth

Fine-tuning a large language model (**LLaMA 3.2 3B Instruct**) for **banking intent classification** on the [BANKING77](https://huggingface.co/datasets/PolyAI/banking77) dataset using **Unsloth** (4-bit QLoRA).

> **Course**: CSC15012 – Applications of Natural Language Processing in Industry
> **Student**: Lê Đình Minh Quân – 23127460
> **University**: HCMUS – VNUHCM

---

## Project Structure

```text
banking-intent-unsloth/
├── scripts/
│   ├── preprocess_data.py    # Download & preprocess BANKING77
│   ├── train.py              # Fine-tune with Unsloth + LoRA
│   └── inference.py          # Standalone inference class
├── configs/
│   ├── train.yaml            # Training hyperparameters
│   └── inference.yaml        # Inference configuration
├── sample_data/
│   ├── train.csv             # Sampled training set
│   ├── test.csv              # Sampled test set
│   └── label_map.json        # Intent label mapping
├── train.sh                  # One-click training script
├── inference.sh              # One-click inference / demo script
├── requirements.txt          # Python dependencies
├── .gitignore
└── README.md
```

---

## Quick Start

### 1. Environment Setup (Google Colab – recommended)

Open a **Colab notebook** with a **GPU runtime** (T4 / A100 / H100) and run:

```bash
# Install Unsloth
pip install unsloth

# Install remaining dependencies
pip install datasets pandas scikit-learn pyyaml

# Clone this repository
git clone https://github.com/{GITHUB_USER}/{REPO_NAME}.git
cd {REPO_NAME}
```

### 2. Data Preparation

```bash
python scripts/preprocess_data.py \\
    --train_samples_per_class 40 \\
    --test_samples_per_class 10 \\
    --output_dir sample_data \\
    --seed 42
```

This downloads the BANKING77 dataset, samples a compute-friendly subset, applies text preprocessing, and saves CSV files under `sample_data/`.

**Current sampled split from this notebook run**
- Train: **{train_rows_text}**
- Test: **{test_rows_text}**
- Number of intent labels: **{num_labels_text}**

### 3. Training

```bash
python scripts/train.py --config configs/train.yaml
```

Or run the all-in-one script:

```bash
bash train.sh
```

**Key hyperparameters** (editable in `configs/train.yaml`):

| Parameter | Value |
|---|---|
| Base model | `unsloth/Llama-3.2-3B-Instruct-bnb-4bit` |
| LoRA rank (r) | 16 |
| LoRA alpha | 16 |
| Learning rate | 2e-4 |
| Optimizer | AdamW 8-bit |
| Batch size | 8 x 4 (gradient accumulation) |
| Epochs | 3 |
| Max sequence length | 512 |
| Precision | bf16 (auto-detects fp16 fallback) |

The fine-tuned adapter is saved to `saved_model/`.

### 4. Inference

**Single message**
```bash
python scripts/inference.py \\
    --config configs/inference.yaml \\
    --message "I want to activate my new card"
```

**Evaluate the full test set**
```bash
python scripts/inference.py --config configs/inference.yaml --evaluate
```

**Interactive mode**
```bash
python scripts/inference.py --config configs/inference.yaml --interactive
```

**Or use the demo shell script**
```bash
bash inference.sh
```

### 5. Inference Class API

```python
from scripts.inference import IntentClassification

classifier = IntentClassification("configs/inference.yaml")
label = classifier("I need to change my PIN")
print(label)  # e.g. "change_pin"
```

---

## Results

| Metric | Value |
|---|---|
| Test Accuracy | {accuracy} |
| Macro Precision | {macro_precision} |
| Macro Recall | {macro_recall} |
| Macro F1 | {macro_f1} |
| Training Loss | {training_loss} |
| Training Time | {training_time} |

The full classification report is saved to `evaluation_results.json` after training.

---

## Links

{notebook_line}

{video_line}

The demo video should clearly show:
1. How the inference script is executed
2. At least one example input message
3. The predicted intent label
4. The final test accuracy

---

## References

- [BANKING77 Dataset](https://huggingface.co/datasets/PolyAI/banking77)
- [Unsloth](https://github.com/unslothai/unsloth)
- [LoRA: Low-Rank Adaptation of Large Language Models](https://arxiv.org/abs/2106.09685)

---

## License

This project is for educational purposes (CSC15012 – HCMUS).
'''
    with open("README.md", "w", encoding="utf-8") as f:
        f.write(readme)
    print("Written: README.md")

with open("configs/train.yaml", "w", encoding="utf-8") as _f:
    _f.write(
        '# ==============================================================================\n'
        '# Training Configuration for Banking77 Intent Classification with Unsloth\n'
        '# ==============================================================================\n\n'
        'model:\n'
        '  name: "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"\n'
        '  max_seq_length: 512\n'
        '  load_in_4bit: true\n'
        '  dtype: null  # auto-detect (float16 for older GPUs, bfloat16 for Ampere+)\n\n'
        'lora:\n'
        '  r: 16\n'
        '  lora_alpha: 16\n'
        '  lora_dropout: 0.0\n'
        '  target_modules:\n'
        '    - q_proj\n'
        '    - k_proj\n'
        '    - v_proj\n'
        '    - o_proj\n'
        '    - gate_proj\n'
        '    - up_proj\n'
        '    - down_proj\n'
        '  bias: "none"\n'
        '  use_gradient_checkpointing: "unsloth"\n'
        '  random_state: 3407\n\n'
        'training:\n'
        '  per_device_train_batch_size: 8\n'
        '  gradient_accumulation_steps: 4\n'
        '  warmup_steps: 10\n'
        '  num_train_epochs: 3\n'
        '  max_steps: -1  # set to positive number to override num_train_epochs\n'
        '  learning_rate: 2.0e-4\n'
        '  weight_decay: 0.01\n'
        '  fp16: false\n'
        '  bf16: true\n'
        '  logging_steps: 10\n'
        '  optim: "adamw_8bit"\n'
        '  lr_scheduler_type: "linear"\n'
        '  seed: 42\n'
        '  output_dir: "outputs"\n'
        '  save_strategy: "epoch"\n'
        '  report_to: "none"\n\n'
        'data:\n'
        '  train_file: "sample_data/train.csv"\n'
        '  test_file: "sample_data/test.csv"\n'
        '  label_map_file: "sample_data/label_map.json"\n\n'
        'save:\n'
        '  model_dir: "saved_model"\n\n'
        'evaluate:\n'
        '  run_eval_after_training: true\n'
        '  max_new_tokens: 64\n'
        '  temperature: 0.0\n'
        '  do_sample: false\n'
        '  results_file: "evaluation_results.json"\n'
    )
print("Written: configs/train.yaml")

with open("configs/inference.yaml", "w", encoding="utf-8") as _f:
    _f.write(
        '# ==============================================================================\n'
        '# Inference Configuration for Banking77 Intent Classification\n'
        '# ==============================================================================\n\n'
        'model:\n'
        '  name: "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"\n'
        '  checkpoint_path: "saved_model"\n'
        '  max_seq_length: 512\n'
        '  load_in_4bit: true\n'
        '  dtype: null\n\n'
        'data:\n'
        '  label_map_file: "sample_data/label_map.json"\n'
        '  test_file: "sample_data/test.csv"\n\n'
        'generation:\n'
        '  max_new_tokens: 64\n'
        '  temperature: 0.0\n'
        '  do_sample: false\n'
    )
print("Written: configs/inference.yaml")

with open("train.sh", "w", encoding="utf-8") as _f:
    _f.write(
        '#!/usr/bin/env bash\n'
        '# ==============================================================================\n'
        '# train.sh – End-to-end pipeline: preprocess data -> fine-tune model\n'
        '# ==============================================================================\n'
        'set -euo pipefail\n\n'
        'echo "=============================================="\n'
        'echo " Banking77 Intent Classification – Training"\n'
        'echo "=============================================="\n\n'
        'echo ""\n'
        'echo "[Step 1/2] Preprocessing BANKING77 dataset ..."\n'
        'python scripts/preprocess_data.py \\\n'
        '    --train_samples_per_class 40 \\\n'
        '    --test_samples_per_class 10 \\\n'
        '    --output_dir sample_data \\\n'
        '    --seed 42\n\n'
        'echo ""\n'
        'echo "[Step 2/2] Fine-tuning model with Unsloth ..."\n'
        'python scripts/train.py --config configs/train.yaml\n\n'
        'echo ""\n'
        'echo "=============================================="\n'
        'echo " Training pipeline complete!"\n'
        'echo "=============================================="\n'
    )
print("Written: train.sh")

with open("inference.sh", "w", encoding="utf-8") as _f:
    _f.write(
        '#!/usr/bin/env bash\n'
        '# ==============================================================================\n'
        '# inference.sh – Run intent classification inference\n'
        '# ==============================================================================\n'
        'set -euo pipefail\n\n'
        'CONFIG="configs/inference.yaml"\n\n'
        'echo "=============================================="\n'
        'echo " Banking77 Intent Classification – Inference"\n'
        'echo "=============================================="\n\n'
        'if [ $# -ge 1 ]; then\n'
        '    MESSAGE="$*"\n'
        '    echo ""\n'
        '    echo "Classifying message: \\"${MESSAGE}\\""\n'
        '    echo ""\n'
        '    python scripts/inference.py --config "$CONFIG" --message "$MESSAGE"\n'
        'else\n'
        '    echo ""\n'
        '    echo "Running demo predictions ..."\n'
        '    echo ""\n'
        '    python scripts/inference.py --config "$CONFIG" --message "I want to activate my new card"\n'
        '    echo "---"\n'
        '    python scripts/inference.py --config "$CONFIG" --message "Why was I charged an extra fee on my statement?"\n'
        '    echo "---"\n'
        '    python scripts/inference.py --config "$CONFIG" --message "My card payment is still pending after 3 days"\n'
        '    echo "---"\n'
        '    python scripts/inference.py --config "$CONFIG" --message "How do I change my PIN number?"\n'
        '    echo "---"\n'
        '    python scripts/inference.py --config "$CONFIG" --message "I lost my phone and need to secure my account"\n'
        '    echo "---"\n'
        '    python scripts/inference.py --config "$CONFIG" --message "What is the exchange rate for USD to EUR?"\n'
        '    echo ""\n'
        '    echo "=============================================="\n'
        '    echo " Running evaluation on test set ..."\n'
        '    echo "=============================================="\n'
        '    python scripts/inference.py --config "$CONFIG" --evaluate\n'
        'fi\n\n'
        'echo ""\n'
        'echo "=============================================="\n'
        'echo " Inference complete!"\n'
        'echo "=============================================="\n'
    )
print("Written: inference.sh")

with open("requirements.txt", "w", encoding="utf-8") as _f:
    _f.write(
        '# ==============================================================================\n'
        '# Banking77 Intent Classification – Dependencies\n'
        '# ==============================================================================\n'
        '# NOTE: On Google Colab, install unsloth first with:\n'
        '#   pip install unsloth\n'
        '# Then install the remaining dependencies below.\n'
        '# ==============================================================================\n\n'
        'unsloth\n'
        'torch>=2.1.0\n'
        'transformers>=4.40.0\n'
        'datasets>=2.18.0\n'
        'trl>=0.8.0\n'
        'peft>=0.10.0\n'
        'bitsandbytes>=0.43.0\n'
        'accelerate>=0.28.0\n'
        'pandas>=2.0.0\n'
        'scikit-learn>=1.3.0\n'
        'pyyaml>=6.0\n'
        'xformers\n'
    )
print("Written: requirements.txt")

with open(".gitignore", "w", encoding="utf-8") as _f:
    _f.write(
        '# Python\n'
        '__pycache__/\n'
        '*.py[cod]\n'
        '*.egg-info/\n\n'
        '# Model checkpoints & outputs (large files – do NOT push to GitHub)\n'
        'saved_model/\n'
        'outputs/\n'
        '*.bin\n'
        '*.safetensors\n\n'
        '# Evaluation results (regenerated on each run)\n'
        'evaluation_results.json\n\n'
        '# Unsloth / Hugging Face caches\n'
        'huggingface_tokenizers_cache/\n'
        'unsloth_compiled_cache/\n'
        'hf_cache/\n'
        '.cache/\n'
        'wandb/\n\n'
        '# IDE / OS\n'
        '.vscode/\n'
        '.idea/\n'
        '.DS_Store\n'
        'Thumbs.db\n\n'
        '# Jupyter\n'
        '.ipynb_checkpoints/\n'
    )
print("Written: .gitignore")

render_readme()

print("\nAll config and utility files created!")


Cloned existing GitHub repo to preserve history.
/content/banking-intent-unsloth
Written: configs/train.yaml
Written: configs/inference.yaml
Written: train.sh
Written: inference.sh
Written: requirements.txt
Written: .gitignore
Written: README.md

All config and utility files created!


## Step 2: Create Python Scripts

The following cells write the three main scripts to disk.

In [3]:
%%writefile scripts/preprocess_data.py
"""
preprocess_data.py
------------------
Download the BANKING77 dataset from HuggingFace, sample a manageable subset,
perform basic text preprocessing, and save train/test splits plus a label map.
"""

import os
import json
import argparse
import re

import pandas as pd
from datasets import load_dataset

BANKING77_LABELS = [
    "activate_my_card", "age_limit", "apple_pay_or_google_pay",
    "atm_support", "automatic_top_up",
    "balance_not_updated_after_bank_transfer",
    "balance_not_updated_after_cheque_or_cash_deposit",
    "beneficiary_not_allowed", "cancel_transfer", "card_about_to_expire",
    "card_acceptance", "card_arrival", "card_delivery_estimate",
    "card_linking", "card_not_working", "card_payment_fee_charged",
    "card_payment_not_recognised", "card_payment_wrong_exchange_rate",
    "card_swallowed", "cash_withdrawal_charge",
    "cash_withdrawal_not_recognised", "change_pin", "compromised_card",
    "contactless_not_working", "country_support", "declined_card_payment",
    "declined_cash_withdrawal", "declined_transfer",
    "direct_debit_payment_not_recognised", "disposable_card_limits",
    "edit_personal_details", "exchange_charge", "exchange_rate",
    "exchange_via_app", "extra_charge_on_statement", "failed_transfer",
    "fiat_currency_support", "freeze_card",
    "get_disposable_virtual_card", "get_physical_card",
    "getting_spare_card", "getting_virtual_card", "lost_or_stolen_card",
    "lost_or_stolen_phone", "order_physical_card", "passcode_forgotten",
    "pending_card_payment", "pending_cash_withdrawal", "pending_top_up",
    "pending_transfer", "pin_blocked", "receiving_money",
    "Refund_not_showing_up", "request_refund", "reverted_card_payment?",
    "supported_cards_and_currencies", "terminate_account",
    "top_up_by_bank_transfer_charge", "top_up_by_card_charge",
    "top_up_by_cash_or_cheque", "top_up_failed", "top_up_limits",
    "top_up_reverted", "topping_up_by_card", "transaction_charged_twice",
    "transfer_fee_charged", "transfer_into_account",
    "transfer_not_received_by_recipient", "transfer_timing",
    "unable_to_verify_identity", "verify_my_identity",
    "verify_source_of_funds", "verify_top_up",
    "virtual_card_not_working", "visa_or_mastercard",
    "why_verify_identity", "wrong_amount_of_cash_received",
    "wrong_exchange_rate_for_cash_withdrawal",
]


def _load_banking77():
    """Load BANKING77 with multiple fallbacks for different datasets versions."""
    # Strategy 1: regular dataset load with trust_remote_code
    try:
        ds = load_dataset("PolyAI/banking77", trust_remote_code=True)
        label_names = ds["train"].features["label"].names
        print("  Loaded via default method (trust_remote_code=True).")
        return ds, label_names
    except Exception as e:
        print(f"  Default load failed: {e}")

    # Strategy 2: auto-converted parquet branch
    try:
        ds = load_dataset(
            "PolyAI/banking77",
            revision="refs/convert/parquet",
            trust_remote_code=True,
        )
        try:
            label_names = ds["train"].features["label"].names
        except (AttributeError, KeyError):
            label_names = BANKING77_LABELS
        print("  Loaded via parquet revision.")
        return ds, label_names
    except Exception as e:
        print(f"  Parquet revision failed: {e}")

    # Strategy 3: direct parquet files from the Hub
    try:
        base = "hf://datasets/PolyAI/banking77@refs%2Fconvert%2Fparquet"
        ds = load_dataset(
            "parquet",
            data_files={
                "train": f"{base}/default/train/0000.parquet",
                "test": f"{base}/default/test/0000.parquet",
            },
        )
        print("  Loaded via direct parquet URLs.")
        return ds, BANKING77_LABELS
    except Exception as e:
        print(f"  Direct parquet failed: {e}")

    raise RuntimeError(
        "Could not load BANKING77. Try upgrading datasets or re-running with internet access."
    )


def clean_text(text: str) -> str:
    """Basic text normalization for banking queries."""
    text = str(text).strip()
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    return text


def _sample_per_class(df: pd.DataFrame, max_per_class: int, seed: int) -> pd.DataFrame:
    if max_per_class <= 0:
        return df.copy()

    sampled = (
        df.groupby("label", group_keys=False)
        .apply(
            lambda g: g.sample(
                n=min(max_per_class, len(g)),
                random_state=seed,
            )
        )
        .reset_index(drop=True)
    )
    return sampled


def main(args):
    print("Loading BANKING77 dataset from HuggingFace ...")
    dataset, label_names = _load_banking77()

    id2label = {str(i): name for i, name in enumerate(label_names)}
    label2id = {name: i for i, name in enumerate(label_names)}
    num_labels = len(label_names)

    train_df = pd.DataFrame(dataset["train"])
    test_df = pd.DataFrame(dataset["test"])

    train_df["label_text"] = train_df["label"].map(lambda x: id2label[str(x)])
    test_df["label_text"] = test_df["label"].map(lambda x: id2label[str(x)])

    original_train_counts = train_df["label"].value_counts().sort_index()
    original_test_counts = test_df["label"].value_counts().sort_index()

    # ---- Sample a subset to fit available compute ----
    train_df = _sample_per_class(train_df, args.train_samples_per_class, args.seed)
    test_df = _sample_per_class(test_df, args.test_samples_per_class, args.seed)

    # ---- Clean text ----
    train_df["text"] = train_df["text"].apply(clean_text)
    test_df["text"] = test_df["text"].apply(clean_text)

    # ---- Shuffle ----
    train_df = train_df.sample(frac=1, random_state=args.seed).reset_index(drop=True)
    test_df = test_df.sample(frac=1, random_state=args.seed).reset_index(drop=True)

    # ---- Save ----
    os.makedirs(args.output_dir, exist_ok=True)

    train_path = os.path.join(args.output_dir, "train.csv")
    test_path = os.path.join(args.output_dir, "test.csv")
    label_map_path = os.path.join(args.output_dir, "label_map.json")

    train_df[["text", "label", "label_text"]].to_csv(train_path, index=False)
    test_df[["text", "label", "label_text"]].to_csv(test_path, index=False)

    with open(label_map_path, "w", encoding="utf-8") as f:
        json.dump(
            {"id2label": id2label, "label2id": label2id, "num_labels": num_labels},
            f,
            indent=2,
            ensure_ascii=False,
        )

    requested_train = args.train_samples_per_class
    requested_test = args.test_samples_per_class

    under_train = int((original_train_counts < requested_train).sum()) if requested_train > 0 else 0
    under_test = int((original_test_counts < requested_test).sum()) if requested_test > 0 else 0

    print(f"Number of intent labels : {num_labels}")
    print(f"Requested train/class   : {requested_train}")
    print(f"Requested test/class    : {requested_test}")
    print(f"Training samples        : {len(train_df)}")
    print(f"Test samples            : {len(test_df)}")
    if requested_train > 0:
        print(f"Classes with < {requested_train} train samples in original split : {under_train}")
    if requested_test > 0:
        print(f"Classes with < {requested_test} test samples in original split  : {under_test}")
    print(f"Files saved to          : {args.output_dir}/")
    print(f"  - {train_path}")
    print(f"  - {test_path}")
    print(f"  - {label_map_path}")


if __name__ == "__main__":
    parser = argparse.ArgumentParser(
        description="Preprocess BANKING77 dataset for intent classification"
    )
    parser.add_argument(
        "--train_samples_per_class",
        type=int,
        default=40,
        help="Max training samples per intent class (0 = use all)",
    )
    parser.add_argument(
        "--test_samples_per_class",
        type=int,
        default=10,
        help="Max test samples per intent class (0 = use all)",
    )
    parser.add_argument(
        "--output_dir",
        type=str,
        default="sample_data",
        help="Directory to save processed CSV files",
    )
    parser.add_argument("--seed", type=int, default=42, help="Random seed")
    args = parser.parse_args()
    main(args)


Overwriting scripts/preprocess_data.py


In [4]:
%%writefile scripts/train.py
"""
train.py
--------
Fine-tune a pre-trained LLM for banking intent classification using Unsloth
(4-bit LoRA) and the HuggingFace SFTTrainer.
"""

import os
import json
import argparse
import time
import warnings
from difflib import get_close_matches

import yaml
import torch
import pandas as pd
from datasets import Dataset
from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import SFTTrainer, SFTConfig
from sklearn.metrics import accuracy_score, classification_report

warnings.filterwarnings(
    "ignore",
    message=r"Both `max_new_tokens` .* `max_length`.*",
    category=UserWarning,
)
warnings.filterwarnings(
    "ignore",
    message=r"The attention mask API under `transformers\.modeling_attn_mask_utils`.*",
    category=FutureWarning,
)

# ---------------------------------------------------------------------------
# Prompt template (Alpaca-style) – must be identical at training & inference
# ---------------------------------------------------------------------------
PROMPT_TEMPLATE = (
    "Below is an instruction that describes a task, paired with an input "
    "that provides further context. Write a response that appropriately "
    "completes the request.\n\n"
    "### Instruction:\n"
    "Classify the intent of the following banking customer message. "
    "Reply with only the intent label, nothing else.\n\n"
    "### Input:\n{text}\n\n"
    "### Response:\n{label}"
)

PROMPT_TEMPLATE_INFERENCE = (
    "Below is an instruction that describes a task, paired with an input "
    "that provides further context. Write a response that appropriately "
    "completes the request.\n\n"
    "### Instruction:\n"
    "Classify the intent of the following banking customer message. "
    "Reply with only the intent label, nothing else.\n\n"
    "### Input:\n{text}\n\n"
    "### Response:\n"
)


def format_dataset(df: pd.DataFrame, tokenizer) -> Dataset:
    """Convert a DataFrame into a HuggingFace Dataset with formatted prompts."""
    eos = tokenizer.eos_token
    records = []
    for _, row in df.iterrows():
        prompt = PROMPT_TEMPLATE.format(text=row["text"], label=row["label_text"])
        records.append({"formatted_text": prompt + eos})
    return Dataset.from_pandas(pd.DataFrame(records))


def evaluate_model(model, tokenizer, test_df, id2label, gen_cfg):
    """Run inference on the test set and return accuracy + classification report."""
    FastLanguageModel.for_inference(model)

    all_labels = list(set(id2label.values()))
    y_true, y_pred = [], []

    print(f"\nEvaluating on {len(test_df)} test samples ...")
    start = time.time()

    for idx, row in test_df.iterrows():
        prompt = PROMPT_TEMPLATE_INFERENCE.format(text=row["text"])
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=gen_cfg.get("max_new_tokens", 64),
                temperature=gen_cfg.get("temperature", 0.0),
                do_sample=gen_cfg.get("do_sample", False),
                use_cache=True,
                pad_token_id=tokenizer.eos_token_id,
            )

        new_tokens = output_ids[0][inputs["input_ids"].shape[1] :]
        response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

        predicted = response.split("\n")[0].strip()

        if predicted not in all_labels:
            matches = get_close_matches(predicted, all_labels, n=1, cutoff=0.4)
            predicted = matches[0] if matches else predicted

        y_true.append(row["label_text"])
        y_pred.append(predicted)

        if (idx + 1) % 50 == 0:
            elapsed = time.time() - start
            print(f"  [{idx+1}/{len(test_df)}]  elapsed {elapsed:.1f}s")

    acc = accuracy_score(y_true, y_pred)
    report_text = classification_report(y_true, y_pred, zero_division=0)
    report_dict = classification_report(y_true, y_pred, zero_division=0, output_dict=True)
    elapsed = time.time() - start

    macro_precision = report_dict["macro avg"]["precision"]
    macro_recall = report_dict["macro avg"]["recall"]
    macro_f1 = report_dict["macro avg"]["f1-score"]

    print(f"\nTest Accuracy : {acc:.4f}  ({sum(a==b for a,b in zip(y_true,y_pred))}/{len(y_true)})")
    print(f"Macro Precision: {macro_precision:.4f}")
    print(f"Macro Recall   : {macro_recall:.4f}")
    print(f"Macro F1       : {macro_f1:.4f}")
    print(f"Evaluation time: {elapsed:.1f}s")
    print(f"\nClassification Report:\n{report_text}")

    return {
        "accuracy": acc,
        "num_correct": sum(a == b for a, b in zip(y_true, y_pred)),
        "num_total": len(y_true),
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
        "classification_report": report_text,
        "predictions": [
            {"text": t, "true": yt, "pred": yp}
            for t, yt, yp in zip(test_df["text"].tolist(), y_true, y_pred)
        ],
    }


def main(config_path: str):
    # ---- Load config ----
    with open(config_path, encoding="utf-8") as f:
        cfg = yaml.safe_load(f)

    model_cfg = cfg["model"]
    lora_cfg = cfg["lora"]
    train_cfg = cfg["training"]
    data_cfg = cfg["data"]
    save_cfg = cfg["save"]
    eval_cfg = cfg.get("evaluate", {})

    # ---- Load preprocessed data ----
    train_df = pd.read_csv(data_cfg["train_file"])
    test_df = pd.read_csv(data_cfg["test_file"])

    with open(data_cfg["label_map_file"], encoding="utf-8") as f:
        label_data = json.load(f)
    id2label = label_data["id2label"]

    print(f"Training samples : {len(train_df)}")
    print(f"Test samples     : {len(test_df)}")
    print(f"Num labels       : {label_data['num_labels']}")

    # ---- Load model ----
    print(f"\nLoading model: {model_cfg['name']} ...")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_cfg["name"],
        max_seq_length=model_cfg["max_seq_length"],
        load_in_4bit=model_cfg["load_in_4bit"],
        dtype=model_cfg.get("dtype"),
    )

    # ---- Apply LoRA adapters ----
    model = FastLanguageModel.get_peft_model(
        model,
        r=lora_cfg["r"],
        target_modules=lora_cfg["target_modules"],
        lora_alpha=lora_cfg["lora_alpha"],
        lora_dropout=lora_cfg["lora_dropout"],
        bias=lora_cfg["bias"],
        use_gradient_checkpointing=lora_cfg["use_gradient_checkpointing"],
        random_state=lora_cfg.get("random_state", 3407),
    )

    # ---- Format training dataset ----
    train_dataset = format_dataset(train_df, tokenizer)
    print(f"Formatted training dataset: {len(train_dataset)} examples")

    # ---- Training arguments (SFTConfig = TrainingArguments + SFT params) ----
    use_bf16 = is_bfloat16_supported()
    sft_config = SFTConfig(
        per_device_train_batch_size=train_cfg["per_device_train_batch_size"],
        gradient_accumulation_steps=train_cfg["gradient_accumulation_steps"],
        warmup_steps=train_cfg["warmup_steps"],
        num_train_epochs=train_cfg["num_train_epochs"],
        max_steps=train_cfg["max_steps"],
        learning_rate=train_cfg["learning_rate"],
        weight_decay=train_cfg["weight_decay"],
        fp16=not use_bf16,
        bf16=use_bf16,
        logging_steps=train_cfg["logging_steps"],
        optim=train_cfg["optim"],
        lr_scheduler_type=train_cfg["lr_scheduler_type"],
        seed=train_cfg["seed"],
        output_dir=train_cfg["output_dir"],
        save_strategy=train_cfg["save_strategy"],
        report_to=train_cfg.get("report_to", "none"),
        dataset_text_field="formatted_text",
        max_seq_length=model_cfg["max_seq_length"],
        dataset_num_proc=2,
        packing=False,
    )

    # ---- Trainer ----
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        args=sft_config,
    )

    # ---- Train ----
    print("\n========== Starting training ==========")
    train_start = time.time()
    trainer_stats = trainer.train()
    train_elapsed = time.time() - train_start
    print(f"Training completed in {train_elapsed:.1f}s")
    print(f"Training loss: {trainer_stats.training_loss:.4f}")

    # ---- Save model ----
    save_dir = save_cfg["model_dir"]
    os.makedirs(save_dir, exist_ok=True)
    model.save_pretrained(save_dir)
    tokenizer.save_pretrained(save_dir)
    print(f"\nModel saved to: {save_dir}/")

    # ---- Evaluate ----
    if eval_cfg.get("run_eval_after_training", True):
        gen_cfg = {
            "max_new_tokens": eval_cfg.get("max_new_tokens", 64),
            "temperature": eval_cfg.get("temperature", 0.0),
            "do_sample": eval_cfg.get("do_sample", False),
        }
        results = evaluate_model(model, tokenizer, test_df, id2label, gen_cfg)

        results_file = eval_cfg.get("results_file", "evaluation_results.json")
        with open(results_file, "w", encoding="utf-8") as f:
            json.dump(
                {
                    "accuracy": results["accuracy"],
                    "num_correct": results["num_correct"],
                    "num_total": results["num_total"],
                    "macro_precision": results["macro_precision"],
                    "macro_recall": results["macro_recall"],
                    "macro_f1": results["macro_f1"],
                    "classification_report": results["classification_report"],
                    "training_loss": trainer_stats.training_loss,
                    "training_time_seconds": train_elapsed,
                },
                f,
                indent=2,
                ensure_ascii=False,
            )
        print(f"\nEvaluation results saved to: {results_file}")

    print("\n========== Done ==========")


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Fine-tune model with Unsloth")
    parser.add_argument(
        "--config",
        type=str,
        default="configs/train.yaml",
        help="Path to training config YAML",
    )
    args = parser.parse_args()
    main(args.config)


Overwriting scripts/train.py


In [5]:
%%writefile scripts/inference.py
"""
inference.py
------------
Standalone inference module for banking intent classification.
Loads a fine-tuned Unsloth/LoRA checkpoint and predicts the intent of a
single text input.

Usage examples:
    # Single message
    python scripts/inference.py --config configs/inference.yaml \
        --message "I need to activate my new card"

    # Evaluate full test set
    python scripts/inference.py --config configs/inference.yaml --evaluate

    # Interactive mode
    python scripts/inference.py --config configs/inference.yaml --interactive
"""

import os
import json
import argparse
import time
import warnings
from difflib import get_close_matches

import yaml
import torch
import pandas as pd
from unsloth import FastLanguageModel
from sklearn.metrics import accuracy_score, classification_report

warnings.filterwarnings(
    "ignore",
    message=r"Both `max_new_tokens` .* `max_length`.*",
    category=UserWarning,
)
warnings.filterwarnings(
    "ignore",
    message=r"The attention mask API under `transformers\.modeling_attn_mask_utils`.*",
    category=FutureWarning,
)

# Must match the template used during training (see scripts/train.py)
PROMPT_TEMPLATE = (
    "Below is an instruction that describes a task, paired with an input "
    "that provides further context. Write a response that appropriately "
    "completes the request.\n\n"
    "### Instruction:\n"
    "Classify the intent of the following banking customer message. "
    "Reply with only the intent label, nothing else.\n\n"
    "### Input:\n{text}\n\n"
    "### Response:\n"
)


class IntentClassification:
    """
    Banking intent classifier that wraps a fine-tuned Unsloth model.

    Parameters
    ----------
    model_path : str
        Path to a YAML configuration file that contains at least the path to
        the saved model checkpoint, label map, and generation settings.
    """

    def __init__(self, model_path: str):
        with open(model_path, encoding="utf-8") as f:
            config = yaml.safe_load(f)

        model_cfg = config["model"]
        data_cfg = config["data"]
        self.gen_cfg = config.get("generation", {})

        checkpoint = model_cfg["checkpoint_path"]
        print(f"Loading model from: {checkpoint} ...")
        self.model, self.tokenizer = FastLanguageModel.from_pretrained(
            model_name=checkpoint,
            max_seq_length=model_cfg.get("max_seq_length", 512),
            load_in_4bit=model_cfg.get("load_in_4bit", True),
            dtype=model_cfg.get("dtype"),
        )
        FastLanguageModel.for_inference(self.model)

        with open(data_cfg["label_map_file"], encoding="utf-8") as f:
            label_data = json.load(f)

        self.id2label = label_data["id2label"]
        self.label2id = label_data["label2id"]
        self.all_labels = list(self.label2id.keys())
        print(f"Loaded {len(self.all_labels)} intent labels.")

    def __call__(self, message: str) -> str:
        """
        Predict the intent label of a banking customer message.

        Parameters
        ----------
        message : str
            A single banking customer query.

        Returns
        -------
        str
            The predicted intent label.
        """
        prompt = PROMPT_TEMPLATE.format(text=message.strip().lower())
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)

        with torch.no_grad():
            output_ids = self.model.generate(
                **inputs,
                max_new_tokens=self.gen_cfg.get("max_new_tokens", 64),
                temperature=self.gen_cfg.get("temperature", 0.0),
                do_sample=self.gen_cfg.get("do_sample", False),
                use_cache=True,
                pad_token_id=self.tokenizer.eos_token_id,
            )

        new_tokens = output_ids[0][inputs["input_ids"].shape[1] :]
        response = self.tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

        predicted_label = response.split("\n")[0].strip()

        if predicted_label not in self.all_labels:
            matches = get_close_matches(
                predicted_label, self.all_labels, n=1, cutoff=0.4
            )
            if matches:
                predicted_label = matches[0]

        return predicted_label


def evaluate(classifier: IntentClassification, test_file: str):
    """Evaluate the classifier on the full test set."""
    test_df = pd.read_csv(test_file)
    y_true, y_pred = [], []

    print(f"\nRunning evaluation on {len(test_df)} test samples ...\n")
    start = time.time()

    for idx, row in test_df.iterrows():
        pred = classifier(row["text"])
        y_true.append(row["label_text"])
        y_pred.append(pred)

        if (idx + 1) % 50 == 0:
            acc_so_far = accuracy_score(y_true, y_pred)
            elapsed = time.time() - start
            print(f"  [{idx+1}/{len(test_df)}]  acc={acc_so_far:.4f}  elapsed={elapsed:.1f}s")

    acc = accuracy_score(y_true, y_pred)
    report = classification_report(y_true, y_pred, zero_division=0)

    print(f"\n{'='*60}")
    print(f"Test Accuracy: {acc:.4f}  ({sum(a==b for a,b in zip(y_true,y_pred))}/{len(y_true)})")
    print(f"{'='*60}")
    print(f"\nClassification Report:\n{report}")
    return acc


def interactive_mode(classifier: IntentClassification):
    """Run an interactive loop for manual testing."""
    print("\n=== Interactive Intent Classification ===")
    print("Type a banking message and press Enter. Type 'quit' to exit.\n")
    while True:
        try:
            message = input("You: ").strip()
        except (EOFError, KeyboardInterrupt):
            break
        if message.lower() in ("quit", "exit", "q"):
            break
        if not message:
            continue
        label = classifier(message)
        print(f"  -> Intent: {label}\n")
    print("Goodbye!")


def main():
    parser = argparse.ArgumentParser(
        description="Banking intent classification – inference"
    )
    parser.add_argument(
        "--config",
        type=str,
        default="configs/inference.yaml",
        help="Path to inference config YAML",
    )
    parser.add_argument(
        "--message",
        type=str,
        default=None,
        help="A single message to classify",
    )
    parser.add_argument(
        "--evaluate",
        action="store_true",
        help="Run evaluation on the test set",
    )
    parser.add_argument(
        "--interactive",
        action="store_true",
        help="Start interactive classification mode",
    )
    args = parser.parse_args()

    classifier = IntentClassification(args.config)

    if args.message:
        label = classifier(args.message)
        print(f"\nMessage : {args.message}")
        print(f"Intent  : {label}")

    if args.evaluate:
        with open(args.config, encoding="utf-8") as f:
            cfg = yaml.safe_load(f)
        evaluate(classifier, cfg["data"]["test_file"])

    if args.interactive:
        interactive_mode(classifier)

    if not args.message and not args.evaluate and not args.interactive:
        print("\nNo action specified. Use --message, --evaluate, or --interactive.")
        print("Example:")
        print('  python scripts/inference.py --config configs/inference.yaml \\')
        print('      --message "I want to cancel my bank transfer"')


if __name__ == "__main__":
    main()


Overwriting scripts/inference.py


## Step 3: Data Preparation

In [6]:
# ============================================================
# Preprocess BANKING77 dataset
# ============================================================
!python scripts/preprocess_data.py \
    --train_samples_per_class 40 \
    --test_samples_per_class 10 \
    --output_dir sample_data \
    --seed 42

import pandas as pd, json
print("\n--- Sample training data ---")
train_df = pd.read_csv("sample_data/train.csv")
test_df = pd.read_csv("sample_data/test.csv")
print(f"Train shape: {train_df.shape}")
print(f"Test shape : {test_df.shape}")
display(train_df.head(10))

print("\nLabel distribution summary:")
label_counts = train_df["label_text"].value_counts()
print(label_counts.describe())

print("\n--- Label map (first 10) ---")
with open("sample_data/label_map.json", encoding="utf-8") as f:
    lm = json.load(f)
for k, v in list(lm["id2label"].items())[:10]:
    print(f"  {k}: {v}")


Loading BANKING77 dataset from HuggingFace ...
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'PolyAI/banking77' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
README.md: 9.78kB [00:00, 14.8MB/s]
banking77.py: 7.17kB [00:00, 20.3MB/s]
  Default load failed: Dataset scripts are no longer supported, but found banking77.py
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'PolyAI/banking77' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
0000.parquet: 100% 298k/298k [00:00<00:00, 12.7MB/s]
0000.parquet: 100% 93.9k/93.9k [00:00<00:00, 3.88MB/s]
Generating train split: 10003 examples [00:00, 1737724.61 examp

,text,label,label_text
0,does the fee for exchange change?,17,card_payment_wrong_exchange_rate
1,i topped up and now my money is not there anym...,61,top_up_reverted
2,my money i had was gone and i could not get gas!,62,topping_up_by_card
3,what other currencies are available to change to?,33,exchange_via_app
4,how can i report that my card was stolen? i ma...,41,lost_or_stolen_card
5,my top up may have been reverted,61,top_up_reverted
6,can i receive transfers from my employer this ...,50,receiving_money
7,"i got some cash at an atm earlier, but now app...",75,wrong_amount_of_cash_received
8,need help to get my card to work.,14,card_not_working
9,why was i charged $1 in a transaction?,34,extra_charge_on_statement



Label distribution summary:
count    77.000000
mean     39.935065
std       0.569803
min      35.000000
25%      40.000000
50%      40.000000
75%      40.000000
max      40.000000
Name: count, dtype: float64

--- Label map (first 10) ---
  0: activate_my_card
  1: age_limit
  2: apple_pay_or_google_pay
  3: atm_support
  4: automatic_top_up
  5: balance_not_updated_after_bank_transfer
  6: balance_not_updated_after_cheque_or_cash_deposit
  7: beneficiary_not_allowed
  8: cancel_transfer
  9: card_about_to_expire


## Step 4: Fine-Tuning with Unsloth

This cell fine-tunes **LLaMA 3.2 3B** with QLoRA. On an H100 this usually takes roughly **5–15 minutes** depending on current package versions and runtime load.


In [7]:
# ============================================================
# Fine-tune with Unsloth
# ============================================================
!python scripts/train.py --config configs/train.yaml


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
Training samples : 3075
Test samples     : 770
Num labels       : 77

Loading model: unsloth/Llama-3.2-3B-Instruct-bnb-4bit ...
==((====))==  Unsloth 2026.4.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA H100 80GB HBM3. Num GPUs = 1. Max memory: 79.179 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
model.safetensors: 100% 2.24G/2.24G [00:03<00:00, 698MB/s] 
Loading weights: 100% 254/254 [00:00<00:00, 640.88it/s]
generation_config.json: 100% 2

## Step 5: Inference & Evaluation

In [8]:
# ============================================================
# Inference demos (loads model ONCE, runs all demos)
# ============================================================
from scripts.inference import IntentClassification

classifier = IntentClassification("configs/inference.yaml")

demo_messages = [
    "I want to activate my new card",
    "Why was I charged an extra fee on my statement?",
    "My card payment is still pending after 3 days",
    "How do I change my PIN number?",
    "I lost my phone and need to secure my account",
    "What is the exchange rate for USD to EUR?",
]

print("=" * 60)
print("  INFERENCE DEMOS")
print("=" * 60)

for msg in demo_messages:
    label = classifier(msg)
    print(f"\nMessage : {msg}")
    print(f"Intent  : {label}")

print("\n" + "=" * 60)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading model from: saved_model ...
==((====))==  Unsloth 2026.4.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA H100 80GB HBM3. Num GPUs = 1. Max memory: 79.179 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-bnb-4bit as a legacy tokenizer.
Unsloth 2026.4.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.
Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Loaded 77 intent labels.
  INFERENCE DEMOS


Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Message : I want to activate my new card
Intent  : activate_my_card

Message : Why was I charged an extra fee on my statement?
Intent  : extra_charge_on_statement


Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Message : My card payment is still pending after 3 days
Intent  : pending_card_payment

Message : How do I change my PIN number?
Intent  : change_pin


Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Message : I lost my phone and need to secure my account
Intent  : lost_or_stolen_phone

Message : What is the exchange rate for USD to EUR?
Intent  : exchange_rate



In [9]:
# ============================================================
# Full test-set evaluation (reuses the model loaded above)
# ============================================================
from scripts.inference import evaluate
import yaml

with open("configs/inference.yaml", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

evaluate(classifier, cfg["data"]["test_file"])

print("\nUpdating README with actual results ...")
render_readme()


Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Running evaluation on 770 test samples ...



Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

  [50/770]  acc=0.8600  elapsed=8.9s


Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

...

  [700/770]  acc=0.8800  elapsed=112.1s


Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

  [750/770]  acc=0.8800  elapsed=119.6s


Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene


Test Accuracy: 0.8792  (677/770)

Classification Report:
                                                  precision    recall  f1-score   support

                           Refund_not_showing_up       0.90      0.90      0.90        10
                                activate_my_card       1.00      0.90      0.95        10
                                       age_limit       1.00      1.00      1.00        10
                         apple_pay_or_google_pay       0.91      1.00      0.95        10
                                     atm_support       1.00      1.00      1.00        10
                                automatic_top_up       0.89      0.80      0.84        10
         balance_not_updated_after_bank_transfer       1.00      1.00      1.00        10
balance_not_updated_after_cheque_or_cash_deposit       0.88      0.70      0.78        10
                         beneficiary_not_allowed       0.71      0.50      0.59        10
                                 cancel_t

## Optional: Fast CLI inference demo (best for video)

This cell executes the standalone inference script exactly from the command line, with **one** example message so your demo stays short and clear.
Use the accuracy output from **Step 5** to show the final test performance.


In [12]:
# ============================================================
# Optional: Run the standalone inference script with ONE message
# ============================================================
!python scripts/inference.py --config configs/inference.yaml --message "I need to change my PIN number"


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading model from: saved_model ...
==((====))==  Unsloth 2026.4.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA H100 80GB HBM3. Num GPUs = 1. Max memory: 79.179 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Loading weights: 100% 254/254 [00:00<00:00, 640.84it/s]
config.json: 1.47kB [00:00, 4.88MB/s]
tokenizer_config.json: 54.7kB [00:00, 111MB/s]
tokenizer.json: 100% 17.2M/17.2M [00:00<00:00, 28.0MB/s]
special_tokens_map.json: 100% 454/454 [0

## Step 6: Push to GitHub

This cell refreshes the README, cleans junk cache folders, and pushes the clean project to GitHub.

In [14]:
# ============================================================
# Create / sync GitHub repo & push clean code
# ============================================================
import os
import shutil
import subprocess
import requests

# Always refresh README one last time before committing
render_readme()

# Remove local junk folders if Unsloth created them inside the repo
for junk in ["huggingface_tokenizers_cache", "unsloth_compiled_cache"]:
    if os.path.exists(junk):
        shutil.rmtree(junk, ignore_errors=True)
        print(f"Removed local junk folder: {junk}")

# Also untrack them if they were previously committed
subprocess.run(
    ["git", "rm", "-r", "--cached", "--ignore-unmatch", "huggingface_tokenizers_cache", "unsloth_compiled_cache"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
)

try:
    from google.colab import userdata
    TOKEN = userdata.get("GITHUB_PAT")
    print("GitHub PAT loaded from Colab Secrets")
except Exception:
    TOKEN = input("Enter your GitHub Personal Access Token: ").strip()

headers = {
    "Authorization": f"token {TOKEN}",
    "Accept": "application/vnd.github.v3+json",
}
resp = requests.post(
    "https://api.github.com/user/repos",
    headers=headers,
    json={
        "name": REPO_NAME,
        "description": "Banking77 intent classification with Unsloth (CSC15012)",
        "private": False,
    },
)

if resp.status_code == 201:
    print(f"Repository created: https://github.com/{GITHUB_USER}/{REPO_NAME}")
elif resp.status_code == 422:
    print(f"Repository already exists: https://github.com/{GITHUB_USER}/{REPO_NAME}")
else:
    print(f"Repo API response: {resp.status_code} - {resp.json()}")

if not os.path.isdir(".git"):
    subprocess.run(["git", "init"], check=True)
    subprocess.run(["git", "checkout", "-B", "main"], check=True)
else:
    subprocess.run(["git", "checkout", "-B", "main"], check=True)

subprocess.run(["git", "config", "user.email", "ledinhminhquan@users.noreply.github.com"], check=True)
subprocess.run(["git", "config", "user.name", "ledinhminhquan"], check=True)

remote_url = f"https://{TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git"
remotes = subprocess.run(["git", "remote"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True).stdout.split()
if "origin" in remotes:
    subprocess.run(["git", "remote", "set-url", "origin", remote_url], check=True)
else:
    subprocess.run(["git", "remote", "add", "origin", remote_url], check=True)

subprocess.run(["git", "add", "."], check=True)

print("\nGit status:")
subprocess.run(["git", "status", "--short"], check=False)

commit_result = subprocess.run(
    ["git", "commit", "-m", "Update Banking77 intent classification project"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

if commit_result.returncode == 0:
    print(commit_result.stdout)
else:
    print(commit_result.stdout.strip() or "Nothing new to commit")

push_cmd = ["git", "push", "-u", "origin", "main"]
push_result = subprocess.run(push_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

if push_result.returncode != 0:
    print("Normal push failed. Retrying with --force-with-lease ...")
    push_cmd = ["git", "push", "-u", "origin", "main", "--force-with-lease"]
    push_result = subprocess.run(push_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

print(push_result.stdout)

print("\nAll code pushed to GitHub!")
print(f"https://github.com/{GITHUB_USER}/{REPO_NAME}")

if VIDEO_LINK == "PASTE_PUBLIC_VIDEO_LINK_HERE":
    print("\nReminder: update VIDEO_LINK in Step 1 and re-run this cell after you upload the public demo video.")


Written: README.md
GitHub PAT loaded from Colab Secrets
Repository already exists: https://github.com/ledinhminhquan/banking-intent-unsloth

Git status:
[main 89f93c0] Update Banking77 intent classification project
 1 file changed, 7 insertions(+), 7 deletions(-)

To https://github.com/ledinhminhquan/banking-intent-unsloth.git
   198dd43..89f93c0  main -> main
Branch 'main' set up to track remote branch 'main' from 'origin'.


All code pushed to GitHub!
https://github.com/ledinhminhquan/banking-intent-unsloth


## All Done!

### Completed automatically
- Project structure created / synced
- BANKING77 data preprocessed
- LLaMA 3.2 3B fine-tuned with QLoRA via Unsloth
- Model evaluated on the test set
- Notebook-level inference demos run
- Optional script-level inference demo available for video recording
- README refreshed with real metrics
- Cache folders cleaned before GitHub push

**GitHub:** https://github.com/ledinhminhquan/banking-intent-unsloth
